## Customer Personality Analysis — EDA & Feature Engineering Summary

Absolutely — here’s a more **descriptive, plain-English write-up** you can paste straight into your report/README. It’s organized so non-experts can follow the logic from data → insight → action. I’ve kept the figure numbers/titles aligned with your images.

---

# Customer Personality Analysis — EDA Narrative (for teammates)

## What this analysis covers

We explored who our customers are, how they buy, and what predicts whether they **respond** to a marketing campaign. We engineered features (RFM, channel shares, category shares, spend ratios, etc.), checked data quality, and then built visuals to tell the story. This document explains each figure in everyday language, why it matters, and how we’ll use it.

---

## Big picture (one paragraph)

Only about **15%** of customers responded to the last campaign, so “send to everyone” wastes budget. People are mostly **30–60** years old with right-skewed income (a few high earners). Spending rises with income, and the customers most likely to respond are **recent** buyers with **moderate to high** purchase frequency. Responders lean a little more toward **catalog/digital** and rely **less on discounts**. Our highest-value customers show a **premium taste profile** (Wines first, then Meat). These patterns tell us which features to feed into the prediction model and suggest clear customer personas for clustering.

---

## Figure-by-figure explanations

### Fig 0 — Campaign Response Distribution

**What you’re looking at:** A simple bar chart: **~15% Responded**, **~85% No Response**.
**Read it like this:** The target is imbalanced; if we predicted “No” for everyone, accuracy would look high but be meaningless.
**Why it matters:** Metrics like **PR-AUC** and **recall@top-k** will be more informative than plain accuracy.
**What we’ll do:** Use **stratified** train/validation/test splits; consider class weights or focal loss so the model doesn’t ignore the minority class.

---

### Fig 1 — Missing Data by Feature

**What you’re looking at:** The percentage of missing values by feature (all low). `F_score` and `Frequency_pm` are the only ones with ~2.2% missing; a few shares/interactions are ~0.3%.
**Read it like this:** The dataset is **clean**; nothing here will block modeling or clustering.
**Why it matters:** Low missingness means simpler imputation (fill with medians or recompute where tenure is zero).
**What we’ll do:**

* Impute `Frequency_pm`/`F_score` using **median**, or set `Frequency_pm=0` where `Tenure_Months=0`.
* For customers with zero purchases, channel shares are set to **0 by design** (they’re undefined otherwise).

---

### Fig 2 — Customer Age Distribution

**What you’re looking at:** Histogram of age (limited to a professional 18–100 range). Most customers cluster **30–60**; mean ≈ **45**, median ≈ **44** (annotation on the plot).
**Read it like this:** We mainly serve working-age adults; there’s still a decent 60+ segment.
**Why it matters:** Age ties to needs and media habits; non-linear effects are likely.
**What we’ll do:** Use **age buckets** (18–29 / 30–44 / 45–59 / 60+) as features and for segment-specific messaging.

---

### Fig 3 — Customer Income Distribution (Log Scale)

**What you’re looking at:** Income after a log transform (log(Income+1)). The distribution is much smoother and closer to bell-shaped.
**Read it like this:** Raw income is heavily skewed; the log scale reveals the “typical” customer better.
**Why it matters:** Many models (and clustering) perform better on log-scaled money variables.
**What we’ll do:** Feed **`log_Income`** to models; also keep **income quintiles (`Income_q`)** for easy segmenting.

---

### Fig 4 — Customer Spending by Education Level

**What you’re looking at:** Boxen plots of **Total_Spent** grouped by Education. Median spend rises from **Basic** through **PhD/Master**, but each group has a wide spread.
**Read it like this:** Education is a **useful but not perfect** proxy for spend potential.
**Why it matters:** Education can help shape personas and creative tone.
**What we’ll do:** Keep **Education** as a categorical predictor; consider different offers/creatives by education band.

---

### Fig 5 — Campaign Response Rate by Age Segment

**What you’re looking at:** The **percentage** of responders in each age bucket. Edges do a bit better: **18–29** and **60+** are slightly higher; **45–59** dips.
**Read it like this:** Younger and older segments are a touch more responsive than mid-age.
**Why it matters:** Tailoring tone and channel by age can lift results.
**What we’ll do:** Keep **`Age_Bin`** in the model; try age-specific subject lines/imagery in tests.

---

### Fig 6 — Income vs. Spending by Campaign Response (log–log)

**What you’re looking at:** A scatter on log scales. Clear positive slope: more income → more spend. Responders overlap non-responders but appear more in the **mid-to-high spend** cloud.
**Read it like this:** Ability to pay + willingness to shop both matter.
**Why it matters:** Normalizing spend by income can surface “over-performers.”
**What we’ll do:** Use **`Total_Spent`**, **`log_Monetary`**, and **`Spending_Ratio = Total_Spent / Income`**. That ratio flags customers who spend a lot relative to their means.

---

### Fig 7 — Feature Correlation Matrix

**What you’re looking at:** Pairwise correlations for numeric features (upper triangle masked to reduce clutter). The **RFM family** (Recency, Frequency, Monetary, R/F/M scores, Total_Purchases) and **channel counts** line up as expected.
**Read it like this:** Some features are **related** (e.g., purchases ↔ monetary), which is normal.
**Why it matters:** Linear models should use regularization; trees handle it naturally.
**What we’ll do:** Standardize numeric features; prefer **regularized logistic/GBMs**; avoid dropping useful signal just because it correlates.

---

### Fig 8 — Channel Preferences by Campaign Response

**What you’re looking at:** Distribution of **Web/Store/Catalog/Deals shares** split by responders vs non-responders. Responders show **slightly higher Catalog share**, **lower Store share**, **lower Deals share**; **Web share** is roughly similar.
**Read it like this:** Likely responders are a bit more **catalog/digital** and **less discount-driven**.
**Why it matters:** Matching channel to audience reduces waste.
**What we’ll do:** For “likely responders,” emphasize **catalog/digital outreach** and avoid heavy discounting.

---

### Fig 9 — Category Spending Patterns (Top 200 Customers)

**What you’re looking at:** Category **shares** among the top 200 spenders. **Wines** dominates, followed by **Meat**; other categories are small.
**Read it like this:** Our highest-value customers have a **premium taste** profile.
**Why it matters:** High-value personas should see premium bundles and content that fits this taste.
**What we’ll do:** Create a persona like **“Premium Gourmand”** (Wines + Meat focus), and test creative that elevates quality over discounts.

---

### Fig 10 — Mean Response by Recency × Frequency (RFM heatmap)

**What you’re looking at:** Average response rate for each **Recency bin** (Hot/Warm/Cold) by **Frequency quintile**. The brightest cells are **Hot (0–30 days)** with **mid–high frequency** (peak around ~0.34).
**Read it like this:** Recency is the **strongest single driver**; frequency adds extra lift.
**Why it matters:** This gives a **targeting rule** even before modeling.
**What we’ll do:** Prioritize customers who bought **recently** and **more than average**; keep **`Recency`**, **`Frequency_pm`**, and **`RFM_Total`** as top predictors.

---

### Fig 11 — Mutual Information with Response

**What you’re looking at:** Features ranked by how much unique signal they carry for `Response`. Top items include **Past_Accepted** (campaign history), **Spending_Ratio**, **Store_Share**, **Monetary**, **Income**, **Catalog_Share**, **Frequency_pm**, **Customer_Tenure_Days**, **Premium_Tilt**, **Category_Concentration**, **Recency**, **RFM_Total**.
**Read it like this:** History + value + engagement + channel mix all matter; each adds information the others don’t fully capture.
**Why it matters:** This is the **short-list for modeling**.
**What we’ll do:** Start with these features; tune with regularization, and keep an eye on interaction terms (e.g., **Age × Web_Share**).

---

## What this means for our two goals

### Goal A — Predict who will respond (classification)

* **Use**: `Recency`, `RFM_Total`, `Frequency_pm`, `Monetary/log_Monetary`, `Past_Accepted`, `Spending_Ratio`, `Customer_Tenure_Days`, `Web/Store/Catalog_Share`, `Category_Concentration`, `Premium_Tilt`, `log_Income`, `Age_Bin`, `Education`.
* **Train/Validate**: Stratified splits; scale numerics (or use trees); try class weights.
* **Judge success**: PR-AUC; **Recall@Top-k** (e.g., top 10% most likely) to simulate campaign targeting.
* **Guardrails**: No leakage (we didn’t engineer any from `Response`), watch multicollinearity for linear models.

### Goal C — Create customer personas (clustering)

* **Inputs (scaled)**: `log_Monetary`, `Frequency_pm`, `Recency`, `{Wines…Gold}_Share`, `Premium_Tilt`, `Category_Concentration`, `Web/Store/Catalog_Share`, `log_Income`, `Age`.
* **What to expect** (from EDA):

  1. **Premium Gourmand** — high Wines/Meat, high spend.
  2. **Catalog-leaning Digital** — higher catalog share, decent response.
  3. **Store-centric Traditional** — higher store share, lower response.
  4. **Deal-sensitive Occasional** — higher Deals share, lower frequency/recency.
* **Evaluate**: silhouette score + cluster stability; label clusters with median profiles.

---

## Practical next steps

1. **Lock features** in `customers_featured.csv` (done) and share with modelers.
2. **Model v1**: regularized logistic + tree model (e.g., LightGBM); compare on PR-AUC + recall@Top-k.
3. **Targeting test**: select top 10–20% by predicted probability; measure uplift vs control.
4. **Personas**: cluster on scaled features above; write one-paragraph descriptions with typical values (age, spend, channels, categories).
5. **Creative/Channel ideas**:

   * Premium Gourmand → wine/meat bundles; catalog/digital first; minimal discounts.
   * Catalog-leaning Digital → send catalog + email follow-up; emphasize convenience.
   * Store-centric → in-store events/coupons; shorter recency windows.
   * Deal-sensitive → limited, targeted promotions; nudge frequency.

---

## Glossary (quick definitions)

* **Recency**: Days since last purchase (smaller = more recent).
* **Frequency_pm**: Purchases per month of tenure.
* **Monetary / log_Monetary**: Lifetime spend; log reduces skew.
* **R/F/M scores**: Quantile scores for Recency/Frequency/Monetary; **RFM_Total** is their sum.
* **Channel shares**: Fraction of purchases via **Web / Store / Catalog / Deals** (sum ≈ 1 when purchases > 0).
* **Category shares**: Fraction of spend in **Wines/Fruits/Meat/Fish/Sweets/Gold** (sum = 1 when spend > 0).
* **Premium_Tilt**: (Wines + Gold) share — a quick “premium” signal.
* **Category_Concentration**: Herfindahl index of category shares (higher = more focused).
* **Spending_Ratio**: Total spend relative to income (normalizes big vs modest earners).
* **Past_Accepted**: # of past campaigns the customer accepted.


